<a href="https://colab.research.google.com/github/georginadd/EMSC2010_major_project/blob/main/EMSC2010_Major_Assignment_u7925896.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EMSC2010 Major Assignment - Investigating the Relationship Between Ocean Temperature and Coral Bleaching in Australian Reefs


# Project Overview
Project title:

**Datasets used (name and source):** Australian Coral Bleaching Multifactor Dataset

Liu, Bailu; Guan, Lei; Foo, Shawna (2026), “Australian Coral Bleaching Multifactor Dataset”, Mendeley Data, V1, doi: 10.17632/57wwr8bvcz.1

**Metadata**:

Australian Coral Bleaching Multifactor Dataset is an analysis-ready tabular dataset of 2,985 quality-controlled coral bleaching observations across Australian reef regions and adjacent seas, spanning 1998–2017. Each record represents a unique 0.05° grid cell and survey month, with geolocation (LAT, LON), time (Year, Month), bleaching severity, and a suite of multi-factor environmental predictors assembled for statistical modelling and benchmarking.

Bleaching outcomes

   Bleaching observations and severity information in this release were sourced from the updated global mass coral bleaching database (Version 2.0) compiled by Virgen-Urcelay & Donner (2023, v2). The original event-level records were subset to the Australian region and then harmonized to the analysis format used here (unique 0.05° grid cell × survey month). Bleaching is provided as an ordinal severity class (Seve2, 0–3) together with a continuous proxy (Seve) mapped to (0,1) for downstream statistical modelling.

Thermal-stress and climate variables

  Using the NOAA Coral Reef Watch (CRW) 5-km CoralTemp v3.1 SST product, we derived multiple thermal-exposure metrics aligned to each survey record, including Degree Heating Weeks (DHW), maximum HotSpot (maxhs), Heating Increase Rate (HIR; daily rate of increase in DHW during the exposure window), and sea-surface temperature anomaly metrics (SSTA, SSTA_SD, SSTA_freq, SSTA_freq_SD).
  Large-scale climate variability is represented by the Oceanic Niño Index (ONI), computed by NOAA CPC from ERSSTv5 as the 3-month running mean SST anomaly in the Niño-3.4 region (5°N–5°S, 170°–120°W).

Optical environment

  To characterize underwater light regime and water clarity, the dataset includes monthly Level-3 MODIS OceanColor composites: photosynthetically available radiation (PAR) and the diffuse attenuation coefficient at 490 nm (Kd₄₉₀).

Cyclone exposure

  Cyclone exposure indicators (cyclone, cyclone_100, cyclone_200) were derived from the Australian Bureau of Meteorology (BoM) best-track database using distance and time windows around each survey record. Following Lugo-Fernández and Gravois (2010), a 200 km radius was used to indicate cyclone impact.

Bathymetry and proximity metrics

  Depth was taken from event records where available; missing values were supplemented using AusBathyTopo (Geoscience Australia, 2017).
  Distance to land (DIST_L) was derived from the NASA OBPG Distance to the Nearest Coastline global grid (based on GMT intermediate coastline) and resampled to 0.05°.
  Given evidence that mangrove-associated environments can mitigate bleaching, distance to mangroves (DIST_M) was computed as distance to the nearest polygon from Global Mangrove Watch v3 (Bunting et al., 2022).

Regional context

  Regional labels are provided via ECOREGION codes and the original site/region field (Site). Near-surface wind is included as an additional environmental covariate.


AIM:


DATA PREPARATION & COLLECTION:

using:


*   time series analysis
*   correlation/regression
*   hypothesis testing
*   bootstrapping


Aims:

Metadata & dataset cleaning info

**Load in and clean dataset**

In [ ]:
import pandas as pd

coral = pd.read_csv("coral_bleaching_multifactor_dataset.csv")

In [ ]:
coral.isnull().sum()
coral.dropna()

,fid,DHW,maxhs,HIR,SSTA,SSTA_SD,SSTA_freq,SSTA_freq_SD,LAT,LON,...,DIST_L,PAR,kd490,ECOREGION,Site,wind,ONI,cyclone_100,cyclone_200,cyclone
1,620,5.337140,2.17,0.242597,2.24839,0.539326,129,42.0465,-18.658700,146.485200,...,0,50.636,0.750,20143,Torres Strait & Great Barrier Reef,3.09635,1.93,0,0,0
6,626,0.445715,1.07,0.111429,1.14724,0.317168,135,40.8670,-20.259600,148.841700,...,0,48.328,0.636,20143,Torres Strait & Great Barrier Reef,4.62524,1.93,0,0,0
7,627,6.812860,1.91,0.101684,2.20207,0.339992,75,26.1701,-24.925000,152.501500,...,2,53.410,1.518,20202,Bundaberg,3.84543,1.93,0,0,0
9,629,5.284290,2.21,0.240195,2.30129,0.492751,146,43.9204,-18.769100,146.630500,...,-1,48.168,0.718,20143,Torres Strait & Great Barrier Reef,4.19190,1.93,0,0,0
14,666,0.294286,1.05,0.147143,1.44484,0.310936,129,41.9580,-20.340000,148.960000,...,1,43.886,0.704,20143,Great Barrier Reef,6.42845,1.44,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2936,35773,7.394280,1.67,0.116643,2.58000,0.788875,122,37.7760,-18.009367,146.146433,...,0,34.032,0.336,20143,Bedarra reef,4.33984,0.30,0,0,0
2941,35852,9.235720,1.87,0.117142,2.68968,0.782725,141,43.1642,-18.569067,146.482017,...,0,31.892,0.998,20143,Palms West reef,4.90142,0.30,0,0,0
2945,35908,7.768570,1.87,0.206352,2.13258,0.863762,121,26.1701,-24.882600,152.490117,...,2,32.056,3.060,20202,"Bundaberg Area, SE Queensland",4.88651,0.30,0,0,0
2947,36302,11.307200,1.72,0.126178,2.71516,0.911274,116,40.9249,-17.162150,146.005750,...,2,27.740,0.710,20143,High West reef,9.19830,0.31,0,0,0


In [ ]:
!pip install bambi #system command to install bambi package

# Import required packages for data handling, plotting, modelling, and model comparison

import numpy as np
import matplotlib.pyplot as plt
import bambi as bmb
import arviz as az
import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.8/237.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 584.2/584.2 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.4/259.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.9/179.9 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: pytensor
    Found existing installation: pytensor 2.38.3
    Uninstalling pytensor-2.38.3:
      Successfully uninstalled pytensor-2.38.3
  Attempting uninstall: arviz
    Found existing installation: arviz 0.22.0
    Uninstalling arviz-0.22.0:
      Successfully uninstalled arviz-0.22.0
  Attempting uninstall: pymc
    Found existing insta

In [ ]:
coral.head()

,fid,DHW,maxhs,HIR,SSTA,SSTA_SD,SSTA_freq,SSTA_freq_SD,LAT,LON,...,DIST_L,PAR,kd490,ECOREGION,Site,wind,ONI,cyclone_100,cyclone_200,cyclone
0,619,3.488580,1.880000,0.183609,2.00000,0.419631,154,47.9862,-18.7502,147.2688,...,54,51.558,0.590,20143,Torres Strait & Great Barrier Reef,5.16486,1.93,0,0,0
1,620,5.337140,2.170000,0.242597,2.24839,0.539326,129,42.0465,-18.6587,146.4852,...,0,50.636,0.750,20143,Torres Strait & Great Barrier Reef,3.09635,1.93,0,0,0
2,621,1.587140,1.250000,0.079357,1.37742,0.347459,132,47.0726,-17.2661,146.3636,...,34,45.124,0.540,20143,Torres Strait & Great Barrier Reef,4.72417,1.93,0,0,0
3,622,1.768570,1.320000,0.093083,1.39517,0.333901,156,47.5125,-17.8847,146.5938,...,44,48.398,0.838,20143,Torres Strait & Great Barrier Reef,4.73477,1.93,0,0,0
4,623,0.144286,0.980001,0.144286,1.05862,0.367917,112,48.1280,-16.7601,147.7259,...,184,56.630,0.276,20150,Torres Strait & Great Barrier Reef,5.59268,1.93,0,0,0


In [ ]:
# Count how many observations are available for each reef

coral["Site"].value_counts()

,count
Site,
Great Barrier Reef,842
Torres Strait & Great Barrier Reef,801
U/N Reef,173
Coral Sea,46
Torres Strait,30
...,...
Sykes Reef,1
One Tree Island Reef,1
Mast Head Reef,1
